In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

weather_df = pd.read_csv('data/philadelphia_weather_2005_to_2025.csv')
weather_df['date'] = pd.to_datetime(weather_df['date'])

weather_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/philadelphia_weather_2005_to_2025.csv'

In [ ]:
winter_yearly = (
    winter_weather
    .groupby("year")[["high","low","rain","snow"]]
    .agg({
        "high":[np.mean, np.min, np.max],
        "low":[np.mean, np.min, np.max],
        "rain":[np.sum],
        "snow":[np.sum]
    })
)
winter_yearly.columns = ["_".join([c[0], c[1]]) for c in winter_yearly.columns]
winter_yearly = winter_yearly.reset_index()


In [ ]:
# Boxplot of average lows by prediction group
groups = ["More winter", "Early spring"]
data = [winter_merged.loc[winter_merged["prediction"]==g, "low_mean"] for g in groups]

plt.figure(figsize=(6,4))
plt.boxplot(data, tick_labels=groups)
plt.title("Do 'Early spring' years have warmer late-winter LOW temps?")
plt.ylabel("Avg low temp (°F) during Feb 2 → 6 weeks")
plt.grid(True, axis="y", alpha=0.3)
plt.show()


In [4]:
# Add "day since Feb 2" index within each year
winter_weather = winter_weather.copy()
winter_weather["day_since_gh"] = (winter_weather["date"] - pd.to_datetime(dict(year=winter_weather["year"], month=2, day=2))).dt.days

# Merge prediction onto daily rows
winter_daily = winter_weather.merge(predict_df[["year","prediction"]], on="year", how="inner")

# Average trajectory by group (mean low temp each day index)
traj = (
    winter_daily
    .groupby(["prediction","day_since_gh"])[["high","low"]]
    .mean()
    .reset_index()
)

# Plot LOW trajectory
plt.figure(figsize=(10,4))
for g in ["More winter", "Early spring"]:
    sub = traj[traj["prediction"]==g]
    plt.plot(sub["day_since_gh"], sub["low"], marker="o", linewidth=2, label=g)

plt.title("Q3: Do 'Early spring' years warm up faster?")
plt.xlabel("Days since Feb 2")
plt.ylabel("Average low temp (°F)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


NameError: name 'winter_weather' is not defined

In [3]:
# Compute a simple slope per year: low ~ day_since_gh
slopes = []
for yr, df_yr in winter_daily.groupby("year"):
    x = df_yr["day_since_gh"].values
    y = df_yr["low"].values
    # polyfit slope
    slope = np.polyfit(x, y, 1)[0]
    pred = df_yr["prediction"].iloc[0]
    slopes.append({"year": yr, "prediction": pred, "warming_slope_low_per_day": slope})

slopes_df = pd.DataFrame(slopes)

slopes_df.groupby("prediction")["warming_slope_low_per_day"].describe().round(3)


NameError: name 'winter_daily' is not defined